# FIN-GUARD: train the LoRA adapter (RQ6) on a free cloud GPU

Works on **Google Colab** (Runtime > Change runtime type > GPU; a free T4 is enough) and on Kaggle (turn on GPU and Internet).
All data is SYNTHETIC. Expect roughly 30-60 minutes on a T4. Nothing here uploads your data anywhere except the model download from Hugging Face.

What it does: clones the repo, downloads Qwen2.5-0.5B-Instruct, trains the adapter, runs the four-arm evaluation, and packs the adapter + report into one zip you download and put back into the repo on your laptop.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Get the code
If the repository is private, create a GitHub token with read access and use `https://<TOKEN>@github.com/...` instead (do not save the token in the notebook), or upload the repo as a zip and unzip it.

In [ ]:
!git clone https://github.com/Ashmit-A-Rawat/FINGUARD-MajorProject.git FINGUARD
%cd FINGUARD
!git log -1 --oneline

## 3. Install (torch is already installed in Colab)

In [ ]:
!pip install -q -e ".[api,data,llm]" sentence-transformers scikit-learn
# quick sanity check that the training code imports
from llm.fine_tuning.train import train, TrainConfig
print("imports ok")

## 4. Download the base model (about 1 GB)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-0.5B-Instruct", local_dir="llm/models/qwen2.5-0.5b-instruct")
!ls llm/models/qwen2.5-0.5b-instruct

## 5. Two-step probe (checks memory and speed before the full run)

In [ ]:
!OMP_NUM_THREADS=1 PYTHONPATH=. python scripts/train_lora.py --max-steps 2 --out /content/probe

## 6. Full training (about 240 examples x 2 epochs)
Prints loss as it goes and writes `llm/fine_tuning/adapters/qwen0.5b-lora-v1/` (adapter) and `train_log.json` (loss curve, timing, dtype).
If you hit an out-of-memory error on a small GPU, restart the runtime and re-run with `--rank 8`.

In [ ]:
!OMP_NUM_THREADS=1 PYTHONPATH=. python scripts/train_lora.py

## 7. Evaluate: base, base+RAG, fine-tuned, fine-tuned+RAG on the 24 held-out cases
Add `--reference-model llm/models/qwen2.5-1.5b-instruct` only after downloading that model too (optional, about 3 GB, slower).

In [ ]:
!OMP_NUM_THREADS=1 PYTHONPATH=. python experiments/llm/run_finetune_eval.py --out evaluation/reports/llm/finetune_eval.json

## 8. Pack the results and download them

In [ ]:
import shutil, os
os.makedirs("results", exist_ok=True)
shutil.copytree("llm/fine_tuning/adapters/qwen0.5b-lora-v1", "results/qwen0.5b-lora-v1", dirs_exist_ok=True)
shutil.copy("evaluation/reports/llm/finetune_eval.json", "results/finetune_eval.json")
shutil.make_archive("finguard_results", "zip", "results")
try:
    from google.colab import files
    files.download("finguard_results.zip")
except ImportError:
    print("Not on Colab: download finguard_results.zip from the file browser.")

## 9. Put the results back in the repo (on your laptop)
Unzip `finguard_results.zip`, then:
- copy the folder `qwen0.5b-lora-v1` to `llm/fine_tuning/adapters/qwen0.5b-lora-v1/`
- copy `finetune_eval.json` to `evaluation/reports/llm/finetune_eval.json`
- run `make results-report` (RQ6 fills in automatically), or tell Claude "adapter is in place" for the write-up.

Note: the free Colab session deletes its files when it ends, so download the zip before closing the tab.